# Task 3 — Correlation Between News Sentiment and Stock Movement

**Objective:** Load financial news headlines and stock price data, compute VADER and TextBlob sentiment scores, align dates to trading days, and quantify the correlation between news sentiment and daily stock returns.

**Sections:**
1. Setup
2. Data loading
3. Normalise dates to trading days
4. Sentiment analysis (VADER + TextBlob)
5. Categorise sentiment
6. Aggregate sentiment by stock and trading day
7. Compute daily stock returns
8. Merge sentiment with stock returns
9. Pearson / Spearman correlation
10. Scatter plot
11. Bar chart by sentiment category
12. Correlation heatmap
13. Lagged correlation analysis
14. Final interpretation

---
## 1. Setup

In [ ]:
import sys
from pathlib import Path
root = Path.cwd().resolve().parent
if str(root) not in sys.path: sys.path.insert(0, str(root))
import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, matplotlib.dates as mdates, seaborn as sns
from scipy.stats import pearsonr, spearmanr
from src.sentiment import score_headlines, categorize_sentiment
from src.indicators import daily_returns
from src.correlation import align_to_trading_day, aggregate_sentiment, merge_sentiment_returns, compute_correlations, lagged_correlation
from src.visualization import sentiment_vs_returns, correlation_heatmap, bar_chart, scatter
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
%matplotlib inline

---
## 2. Data Loading

We load news headlines and stock price data with synthetic fallback for a fully runnable notebook.

In [ ]:
headlines_path = root / "data" / "raw" / "headlines.csv"
if headlines_path.exists():
    headlines = pd.read_csv(headlines_path, parse_dates=["date"])
    print(f"Loaded headlines: {headlines.shape[0]} rows")
else:
    print("No headlines.csv. Generating synthetic demo data...")
    np.random.seed(42)
    d = pd.date_range("2023-06-01", "2024-12-31", freq="h")
    n = min(len(d), 5000)
    chosen = np.random.choice(d, n, replace=False)
    pos = ["Strong earnings beat estimates", "Record revenue announced", "Stock upgraded by analyst"]
    neg = ["Profit warning issued", "Earnings miss forecasts", "Stock downgraded"]
    neu = ["Annual meeting held", "Quarterly dividend declared", "New office opened"]
    all_h = pos + neg + neu
    stocks = ["AAPL","MSFT","GOOGL","AMZN","META","TSLA"]
    rows = []
    for dt in sorted(chosen):
        rows.append({"headline": np.random.choice(all_h), "publisher": np.random.choice(["Reuters","Bloomberg","CNBC"]), "date": pd.Timestamp(dt).isoformat(), "stock": np.random.choice(stocks)})
    headlines = pd.DataFrame(rows)
    headlines["date"] = pd.to_datetime(headlines["date"])
    print(f"Synthetic headlines: {headlines.shape[0]} rows")
headlines.head(3)

In [ ]:
prices_path = root / "data" / "raw" / "stock_prices.csv"
if prices_path.exists():
    prices = pd.read_csv(prices_path, index_col=0, parse_dates=True)
    print(f"Loaded prices: {prices.shape[0]} rows")
else:
    import yfinance as yf
    raw = yf.download("AAPL", start="2023-01-01", progress=False)
    if isinstance(raw.columns, pd.MultiIndex): raw.columns = raw.columns.get_level_values(0)
    prices = raw[["Close"]]
    prices.columns = ["AAPL"]
    print(f"Downloaded AAPL: {len(prices)} rows")
prices.head(3)

---
## 3. Normalise Dates to Trading Days

Article timestamps may fall on weekends/holidays. We shift dates forward to the next trading day.

In [ ]:
trading_days = prices.index.sort_values()
headlines["date_utc"] = pd.to_datetime(headlines["date"])
headlines["trading_date"] = align_to_trading_day(headlines["date_utc"], trading_days)
n_shifted = (headlines["trading_date"] != headlines["date_utc"].dt.normalize()).sum()
print(f"Shifted to next trading day: {n_shifted} / {len(headlines)}")

---
## 4. Sentiment Analysis (VADER + TextBlob)

We score each headline using VADER (compound -1 to 1) and TextBlob (polarity -1 to 1).

In [ ]:
headlines = score_headlines(headlines, text_column="headline")
headlines[["headline","sentiment_vader","sentiment_textblob"]].sample(8, random_state=1)

**Insight:** VADER and TextBlob scores are broadly correlated but use different lexicons.

---
## 5. Categorise Sentiment

Convert VADER scores into positive (>=0.05), neutral, negative (<=-0.05).

In [ ]:
headlines = categorize_sentiment(headlines, score_col="sentiment_vader")
headlines["sentiment_category"].value_counts().sort_index()

---
## 6. Aggregate Sentiment by Stock and Trading Day

Multiple articles per stock per day are averaged into a single score.

In [ ]:
daily_sent = aggregate_sentiment(headlines, date_col="trading_date", group_col="stock")
daily_sent.head(10)

---
## 7. Compute Daily Stock Returns

Return_t = ((Close_t - Close_t-1) / Close_t-1) * 100

In [ ]:
returns = prices.apply(daily_returns).dropna()
returns.head(5)

---
## 8. Merge Sentiment with Stock Returns

Inner-join daily sentiment with daily returns on aligned trading dates.

In [ ]:
stock_list = headlines["stock"].unique() if "stock" in headlines.columns else ["AAPL"]
merged_dfs = []
for ticker in stock_list[:3]:
    try:
        s = daily_sent.xs(ticker, level="stock")
    except KeyError:
        s = daily_sent
    if ticker not in returns.columns: continue
    r = returns[[ticker]].rename(columns={ticker: "return"})
    m = merge_sentiment_returns(s, r, sentiment_date_col="trading_date")
    m["stock"] = ticker; merged_dfs.append(m)
combined = pd.concat(merged_dfs) if merged_dfs else merge_sentiment_returns(daily_sent, returns, sentiment_date_col="trading_date")
if "stock" not in combined.columns: combined["stock"] = stock_list[0]
print(f"Merged: {len(combined)} rows"); combined.head()

---
## 9. Pearson / Spearman Correlation

Pearson (linear) and Spearman (rank) coefficients for each sentiment/return pair.

In [ ]:
corr_report = compute_correlations(combined, sentiment_cols=["sentiment_vader","sentiment_textblob"], target_cols=["return"])
corr_report

**Interpretation:** Low |r| values are expected: news is only one of many price drivers.

---
## 10. Scatter Plot: Sentiment vs. Returns

In [ ]:
fig = scatter(combined, x="sentiment_vader", y="return",
    title="VADER Sentiment vs. Daily Return",
    xlabel="VADER Score", ylabel="Daily Return", fit_reg=True)
plt.show()

---
## 11. Bar Chart by Sentiment Category

Mean daily return grouped by sentiment category with std error bars.

In [ ]:
combined["category"] = pd.cut(combined["sentiment_vader"], bins=[-1.1,-0.05,0.05,1.1], labels=["negative","neutral","positive"])
agg = combined.groupby("category", observed=True)["return"].agg(["mean","std","count"]).reset_index()
fig = bar_chart(agg, x="category", y="mean", title="Mean Return by Sentiment Category", xlabel="Category", ylabel="Mean Return")
ax = fig.axes[0]
ax.errorbar(agg["category"], agg["mean"], yerr=agg["std"], fmt="none", c="black", capsize=4)
plt.show(); agg

---
## 12. Correlation Heatmap

In [ ]:
fig = correlation_heatmap(combined, title="Correlation Matrix: Sentiment + Returns")
plt.show()

---
## 13. Lagged Correlation Analysis

Testing whether sentiment predicts future returns at lags 0 to 5 trading days.

In [ ]:
lagged = lagged_correlation(combined.reset_index(), sentiment_col="sentiment_vader", return_col="return", max_lag=5)
lagged

In [ ]:
fig, ax = plt.subplots(figsize=(8,4))
ax.plot(lagged["lag_days"], lagged["pearson_r"], marker="o", linewidth=1.5, color="steelblue")
ax.axhline(0, color="gray", linestyle="--", linewidth=0.7)
ax.set(title="Pearson r by Lag (Sentiment leads Returns)", xlabel="Lag (days)", ylabel="Pearson r")
ax.fill_between(lagged["lag_days"], lagged["pearson_r"], 0, alpha=0.1, color="steelblue")
sns.despine(); plt.show()

**Interpretation:** Lag 0 = same-day reaction. Lags 1-5 = next-day to next-week effects. Peak at lag 0 supports semi-strong market efficiency.

---
## 14. Final Interpretation

### Observed Correlation Strength
Pearson/Spearman coefficients for news sentiment vs. daily returns typically fall
in |r| in [0.02, 0.15] — statistically significant but modest. News is only one
of many price drivers.

### Limitations
1. **Generic lexicons** — VADER/TextBlob miss financial jargon. FinBERT would
   improve accuracy.
2. **Daily aggregation** — Intraday effects are lost.
3. **Confounders** — Macro events, Fed policy, sector trends not controlled.
4. **Synthetic data** — Demo correlations are essentially random.

### Lag Effects
If the strongest correlation appears at lag 1+, there is evidence of delayed
reaction or momentum. A peak at lag 0 supports semi-strong market efficiency.

### Next Steps
- Use real financial news data.
- Apply FinBERT for domain-specific scoring.
- Include control variables (VIX, market beta).
- Explore intraday frequency.